# Preprocesado

# 1. Carga y visualización de datos

Se carga el dataset de entrenamiento (`train.csv`) utilizando pandas.

Se realizará el split antes del feature engineering para evitar data leakage en la creación de variables dependientes de distribución.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Cargamos dataset original
train = pd.read_csv("titanic_datasets/unprocessed/train.csv")

# Separamos variables predictoras y target
X = train.drop(columns=["Survived"])
y = train["Survived"]

# Split 80/20 para evaluación real del modelo
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Reconstruimos dataframes de trabajo
train_df = X_train.copy()
test_df = X_test.copy()

train_df["Survived"] = y_train
test_df["Survived"] = y_test

train_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
692,693,3,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,NaN,S,1
481,482,2,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,NaN,S,0
527,528,1,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,0
855,856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,NaN,S,1
801,802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,NaN,S,1


In [3]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 712 entries, 692 to 507
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Pclass       712 non-null    int64  
 2   Name         712 non-null    str    
 3   Sex          712 non-null    str    
 4   Age          575 non-null    float64
 5   SibSp        712 non-null    int64  
 6   Parch        712 non-null    int64  
 7   Ticket       712 non-null    str    
 8   Fare         712 non-null    float64
 9   Cabin        160 non-null    str    
 10  Embarked     710 non-null    str    
 11  Survived     712 non-null    int64  
dtypes: float64(2), int64(5), str(5)
memory usage: 72.3 KB


# 2. Feature Engineering

En esta fase se crean nuevas variables (features) a partir de las existentes.

Objetivo:
- Extraer información relevante no explícita
- Mejorar la capacidad predictiva del modelo

## Title

Vamos a extraer el título de cada pasajero a partir del nombre.

Ejemplo:
- "Braund, Mr. Owen Harris" → "Mr"

El título aporta información relevante sobre:
- Sexo
- Edad aproximada
- Estatus social

También se agrupan títulos poco frecuentes en la categoría "Rare" para evitar ruido en el modelo.

Además, vamos a normalizar títulos equivalentes de diferentes idiomas:
- Mlle, Ms → Miss
- Mme → Mrs

In [4]:
def extract_title(df):
    # Extraemos el título desde el nombre
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

    # Agrupamos títulos raros en una sola categoría
    df["Title"] = df["Title"].replace([
        "Lady", "Countess", "Capt", "Col", "Don", "Dr",
        "Major", "Rev", "Sir", "Jonkheer", "Dona"
    ], "Rare")

    # Normalizamos variantes del mismo título
    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    return df

train_df = extract_title(train_df)
test_df = extract_title(test_df)

# Verificamos que se han extraído los títulos correctamente
print(train_df["Title"].value_counts())
print(test_df["Title"].value_counts())

Title
Mr        412
Miss      144
Mrs       107
Master     31
Rare       18
Name: count, dtype: int64
Title
Mr        105
Miss       41
Mrs        19
Master      9
Rare        5
Name: count, dtype: int64


## Tamaño de familia

Se crea la variable `FamilySize`:

FamilySize = SibSp + Parch + 1

Esto representa el número total de familiares a bordo (incluyendo al propio pasajero).

Además del tamaño exacto de familia, vamos a crear una categorización de si la familia es pequeña, mediana o grande.

In [5]:
def add_family_features(df):
    # Tamaño total de familia a bordo
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

    # Categoría de tamaño familiar
    def categorize(size):
        if size == 1:
            return "Alone"
        elif 2 <= size <= 4:
            return "Small"
        else:
            return "Large"

    df["FamilyCategory"] = df["FamilySize"].apply(categorize)
    return df

train_df = add_family_features(train_df)
test_df = add_family_features(test_df)

# Verificamos las nuevas características
print(train_df[["FamilySize", "FamilyCategory"]].head())

     FamilySize FamilyCategory
692           1          Alone
481           1          Alone
527           1          Alone
855           2          Small
801           3          Small


## Deck (cubierta del barco)

Se extrae la primera letra de la variable `Cabin`, que indica la cubierta.

Ejemplo:
- "C85" → "C"

Dado que hay muchos valores nulos, se rellenan como:
- "U" (Unknown)

Queremos extraer este dato ya que la cubierta puede estar relacionada con:
- Clase social
- Ubicación en el barco

In [6]:
def add_deck(df):
    # Extraemos la cubierta desde Cabin
    df["Deck"] = df["Cabin"].str[0].fillna("Unknown")
    return df

train_df = add_deck(train_df)
test_df = add_deck(test_df)

## Ticket Group

Se calcula el número de pasajeros que comparten el mismo ticket.

Esto permite identificar grupos de personas que viajaban juntas.

La variable `TicketGroup` representa:
- Tamaño del grupo asociado al ticket

Puede aportar información similar a `FamilySize`, pero capturando relaciones no familiares.

Se utiliza únicamente la distribución del conjunto de entrenamiento para evitar filtración de información del conjunto de test.

In [7]:
# frecuencia en TRAIN
ticket_counts = train_df["Ticket"].value_counts()

# train
train_df["TicketGroup"] = train_df["Ticket"].map(ticket_counts)

# test
test_df["TicketGroup"] = test_df["Ticket"].map(ticket_counts)

# los tickets nuevos en test se rellenan con 1 (viajero solo)
test_df["TicketGroup"] = test_df["TicketGroup"].fillna(1)

# seguridad train
train_df["TicketGroup"] = train_df["TicketGroup"].fillna(1)

## Age group

La variable Age es continua. Una relación lineal pura asume que el cambio en la probabilidad de supervivencia es constante por cada año que pasa. Sin embargo, tener 5 años frente a 15 años marcaba una diferencia abismal, pero tener 35 frente a 45 probablemente no. Por eso, vamos a agrupar las edades en categorías para ayudar al modelo a capturar esta no linealidad.

Antes de crear las categorías, como en la columna Age hay muchos valores nulos y no los queremos arrastrar a esta nueva variable, vamos a tratar esos valores.

In [8]:
# =========================================================
# 1. ESTUDIO INICIAL DE LA VARIABLE AGE (TRAIN)
# =========================================================
print("--- 1. ANTES DE LA IMPUTACIÓN (TRAIN) ---")

nulos_age_antes = train_df["Age"].isnull().sum()
porcentaje_nulos = (nulos_age_antes / len(train_df)) * 100

print(f"Nulos en Age (train): {nulos_age_antes} ({porcentaje_nulos:.2f}%)")
print(f"Media de edad (train): {train_df['Age'].mean():.2f} años")
print(f"Desviación estándar (train): {train_df['Age'].std():.2f}\n")


# =========================================================
# 2. ANÁLISIS DE LA ESTRATEGIA DE IMPUTACIÓN
# =========================================================
print("--- 2. MEDIANAS CALCULADAS POR CLASE Y TÍTULO (TRAIN) ---")

tabla_medianas = train_df.groupby(["Pclass", "Title"])["Age"].median().unstack()
print(tabla_medianas)

print("\n" + "-"*40 + "\n")


# =========================================================
# 3. APLICACIÓN DE IMPUTACIÓN (TRAIN → FIT BASE)
# =========================================================
# Imputamos Age en train usando su propia distribución
train_df["Age"] = train_df.groupby(["Pclass", "Title"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

# Red de seguridad
train_df["Age"] = train_df["Age"].fillna(train_df["Age"].median())


# =========================================================
# 4. APLICACIÓN DE IMPUTACIÓN (TEST → USANDO TRAIN)
# =========================================================
age_median = train_df["Age"].median()

test_df["Age"] = test_df.groupby(["Pclass", "Title"])["Age"].transform(
    lambda x: x.fillna(age_median)
)

test_df["Age"] = test_df["Age"].fillna(age_median)


# =========================================================
# 5. COMPROBACIÓN DEL IMPACTO (TRAIN)
# =========================================================
print("--- 3. DESPUÉS DE LA IMPUTACIÓN (TRAIN) ---")

nulos_age_despues = train_df["Age"].isnull().sum()

print(f"Nulos en Age restantes (train): {nulos_age_despues}")
print(f"Media de edad (train): {train_df['Age'].mean():.2f} años")
print(f"Desviación estándar (train): {train_df['Age'].std():.2f}")


# =========================================================
# 6. CREACIÓN DE GRUPOS DE EDAD (AMBOS)
# =========================================================
bins = [0, 12, 18, 60, 120]
labels = ["Child", "Teenager", "Adult", "Elderly"]

train_df["AgeGroup"] = pd.cut(train_df["Age"], bins=bins, labels=labels)
test_df["AgeGroup"] = pd.cut(test_df["Age"], bins=bins, labels=labels)


# =========================================================
# 7. DISTRIBUCIÓN DE GRUPOS DE EDAD (TRAIN)
# =========================================================
print("--- 4. DISTRIBUCIÓN DE GRUPOS DE EDAD (TRAIN) ---")
print(train_df["AgeGroup"].value_counts(dropna=False))

--- 1. ANTES DE LA IMPUTACIÓN (TRAIN) ---
Nulos en Age (train): 137 (19.24%)
Media de edad (train): 29.81 años
Desviación estándar (train): 14.49

--- 2. MEDIANAS CALCULADAS POR CLASE Y TÍTULO (TRAIN) ---
Title   Master  Miss    Mr   Mrs  Rare
Pclass                                
1         2.46  30.0  40.0  39.0  49.0
2         1.00  24.0  30.0  33.5  51.0
3         5.00  18.0  27.0  31.0   NaN

----------------------------------------

--- 3. DESPUÉS DE LA IMPUTACIÓN (TRAIN) ---
Nulos en Age restantes (train): 0
Media de edad (train): 29.40 años
Desviación estándar (train): 13.46
--- 4. DISTRIBUCIÓN DE GRUPOS DE EDAD (TRAIN) ---
AgeGroup
Adult       561
Teenager     77
Child        58
Elderly      16
Name: count, dtype: int64


Podemos observar que la media y desviación estándar apenas han cambiado, por lo que la imputación ha sido un éxito. De este modo, los valores nulos no han sido arrastrados a la nueva variable.

## Verificación de variables

In [9]:
train_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived,Title,FamilySize,FamilyCategory,Deck,TicketGroup,AgeGroup
692,693,3,"Lam, Mr. Ali",male,27.0,0,0,1601,56.4958,NaN,S,1,Mr,1,Alone,Unknown,6,Adult
481,482,2,"Frost, Mr. Anthony Wood ""Archie""",male,30.0,0,0,239854,0.0000,NaN,S,0,Mr,1,Alone,Unknown,1,Adult
527,528,1,"Farthing, Mr. John",male,40.0,0,0,PC 17483,221.7792,C95,S,0,Mr,1,Alone,C,1,Adult
855,856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,NaN,S,1,Mrs,2,Small,Unknown,1,Teenager
801,802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,NaN,S,1,Mrs,3,Small,Unknown,2,Adult


# 3. Gestión de valores nulos

In [10]:
print("--- 1. ANÁLISIS DE NULOS (TRAIN) ---")

total_nulos = train_df.isnull().sum()
porcentaje_nulos = (train_df.isnull().sum() / len(train_df)) * 100

nulos_df = pd.DataFrame({
    "Total Nulos": total_nulos,
    "Porcentaje Nulos (%)": porcentaje_nulos
})

print(nulos_df)

--- 1. ANÁLISIS DE NULOS (TRAIN) ---
                Total Nulos  Porcentaje Nulos (%)
PassengerId               0              0.000000
Pclass                    0              0.000000
Name                      0              0.000000
Sex                       0              0.000000
Age                       0              0.000000
SibSp                     0              0.000000
Parch                     0              0.000000
Ticket                    0              0.000000
Fare                      0              0.000000
Cabin                   552             77.528090
Embarked                  2              0.280899
Survived                  0              0.000000
Title                     0              0.000000
FamilySize                0              0.000000
FamilyCategory            0              0.000000
Deck                      0              0.000000
TicketGroup               0              0.000000
AgeGroup                  0              0.000000


Como podemos observar, los valores nulos se concentran en las variables Embarked y Cabin.

- Embarked: Solo falta la información de dos pasajeros. Al tratarse de una variable categórica (puertos C, Q, S), lo más lógico es asumir que subieron en el puerto con mayor afluencia. Por ello, imputaremos estos nulos utilizando la moda.

- Cabin: Falta casi el 80% de la información. Inventar o imputar una cantidad tan grande de datos introduciría un ruido que perjudicaría gravemente al modelo, por lo que la decisión más sensata es eliminar esta columna. Afortunadamente, en la etapa de Feature Engineering ya extrajimos el valor realmente útil de esta variable: la cubierta del barco, guardada en la variable Deck (rellenando los nulos con "Unknown"). El número exacto de camarote es un dato demasiado disperso y con demasiadas ausencias como para aportar valor predictivo.

Además de cabin, hay columnas que no influyen en nuestro estudio, como PassengerId, Name y Ticket, por lo que vamos a eliminalas también.

In [11]:
# =========================================================
# 1. IMPUTACIÓN DE VARIABLES 
# =========================================================

# Embarked
embarked_mode = train_df["Embarked"].mode()[0]
train_df["Embarked"] = train_df["Embarked"].fillna(embarked_mode)
test_df["Embarked"] = test_df["Embarked"].fillna(embarked_mode)


# =========================================================
# 2. ELIMINACIÓN DE COLUMNAS INNECESARIAS
# =========================================================

cols_to_drop = ["PassengerId", "Name", "Ticket", "Cabin"]

train_df = train_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop)


# =========================================================
# 3. VERIFICACIÓN FINAL DE NULOS
# =========================================================
print("--- 2. VERIFICACIÓN FINAL DE NULOS ---")
print(train_df.drop(columns=["Survived"]).isnull().sum())
print(test_df.drop(columns=["Survived"]).isnull().sum())

--- 2. VERIFICACIÓN FINAL DE NULOS ---
Pclass            0
Sex               0
Age               0
SibSp             0
Parch             0
Fare              0
Embarked          0
Title             0
FamilySize        0
FamilyCategory    0
Deck              0
TicketGroup       0
AgeGroup          0
dtype: int64
Pclass            0
Sex               0
Age               0
SibSp             0
Parch             0
Fare              0
Embarked          0
Title             0
FamilySize        0
FamilyCategory    0
Deck              0
TicketGroup       0
AgeGroup          0
dtype: int64


# 4. One-Hot-Encoding

Vamos a transformar las columnas de texto en columnas de ceros y unos.

Usaremos el parámetro `drop_first=True`. Esto nos ayuda a evitar la multicolinealidad entre variables.
Por ejemplo, si creamos una columna Sex_male (1 si es hombre, 0 si no lo es), el modelo ya sabe implícitamente que si es 0, es mujer. No necesitamos una columna Sex_female adicional, ya que sería información redundante que podría perjudicar al entrenamiento.

Se concatenan temporalmente ambos datasets para garantizar que el One-Hot Encoding genera exactamente las mismas columnas en train y test.

In [12]:
# Convertimos variables categóricas en variables numéricas binarias

categorical_cols = ["Sex", "Embarked", "Title", "Deck", "FamilyCategory", "AgeGroup"]

# Unimos temporalmente para asegurar mismas columnas en train y test
full = pd.concat([train_df, test_df], axis=0)

full = pd.get_dummies(full, columns=categorical_cols, drop_first=True, dtype=int)

# Separamos de nuevo train y test
train_df = full.iloc[:len(train_df)]
test_df = full.iloc[len(train_df):]

# Verificamos ambos datasets
train_df.head()


,Pclass,Age,SibSp,Parch,Fare,Survived,FamilySize,TicketGroup,Sex_male,Embarked_Q,...,Deck_E,Deck_F,Deck_G,Deck_T,Deck_Unknown,FamilyCategory_Large,FamilyCategory_Small,AgeGroup_Teenager,AgeGroup_Adult,AgeGroup_Elderly
692,3,27.0,0,0,56.4958,1,1,6.0,1,0,...,0,0,0,0,1,0,0,0,1,0
481,2,30.0,0,0,0.0000,0,1,1.0,1,0,...,0,0,0,0,1,0,0,0,1,0
527,1,40.0,0,0,221.7792,0,1,1.0,1,0,...,0,0,0,0,0,0,0,0,1,0
855,3,18.0,0,1,9.3500,1,2,1.0,0,0,...,0,0,0,0,1,0,1,1,0,0
801,2,31.0,1,1,26.2500,1,3,2.0,0,0,...,0,0,0,0,1,0,1,0,1,0


In [13]:
test_df.head()

,Pclass,Age,SibSp,Parch,Fare,Survived,FamilySize,TicketGroup,Sex_male,Embarked_Q,...,Deck_E,Deck_F,Deck_G,Deck_T,Deck_Unknown,FamilyCategory_Large,FamilyCategory_Small,AgeGroup_Teenager,AgeGroup_Adult,AgeGroup_Elderly
565,3,24.0,2,0,24.1500,0,3,1.0,1,0,...,0,0,0,0,1,0,1,0,1,0
160,3,44.0,0,1,16.1000,0,2,1.0,1,0,...,0,0,0,0,1,0,1,0,1,0
553,3,22.0,0,0,7.2250,1,1,1.0,1,0,...,0,0,0,0,1,0,0,0,1,0
860,3,41.0,2,0,14.1083,0,3,1.0,1,0,...,0,0,0,0,1,0,1,0,1,0
241,3,27.0,1,0,15.5000,1,2,1.0,0,1,...,0,0,0,0,1,0,1,0,1,0


# 5. Feature Scaling

A continuación, vamos a aplicar un escalado a nuestras variables numéricas utilizando `StandardScaler`.

Este paso es necesario porque la red neuronal aprende mediante Descenso del Gradiente. Si introducimos datos con escalas muy diferentes (por ejemplo, Fare con valores de cientos frente a Parch con valores de unidades), el descenso del gradiente se vuelve inestable, lento y le cuesta converger hacia la solución óptima.

Al estandarizar todas las características numéricas para que tengan una media de 0 y una desviación estándar de 1, garantizamos que el algoritmo trabaje sobre un espacio equilibrado, converja rápidamente y no le dé una importancia falsa a una variable simplemente porque sus números son más grandes.

In [14]:
# Variables numéricas a estandarizar
num_cols = ["Age", "Fare", "SibSp", "Parch", "FamilySize", "TicketGroup"]

scaler = StandardScaler()

# Ajustamos scaler solo con train
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])

# Aplicamos transformación al test
test_df[num_cols] = scaler.transform(test_df[num_cols])

In [15]:
# Separamos variables predictoras y variable objetivo

X_train = train_df.drop(columns=["Survived"])
y_train = train_df["Survived"]

X_test = test_df.drop(columns=["Survived"])
y_test = test_df["Survived"]

# Importación final a csv

In [ ]:
# Dataset de entrenamiento
train_final = X_train.copy()
train_final["Survived"] = y_train
train_final.to_csv("titanic_datasets/processed/train_processed.csv", index=False)

# Dataset de validación/test interno
test_final = X_test.copy()
test_final["Survived"] = y_test
test_final.to_csv("titanic_datasets/processed/test_processed.csv", index=False)